In [1]:
#Homework: Agentic RAG

In [ ]:
import os
from dotenv import load_dotenv
from google import genai  # Tu librería oficial moderna de Google

# 1. Cargamos las variables de tu archivo .env
load_dotenv()

# 2. Inicializamos tu cliente oficial de Google Gemini
gemini_client = genai.Client()

from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [ ]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [24]:
#Q1. How many lesson pages
#How many lesson pages are in the dataset?

In [ ]:
print(len(documents))

72


In [27]:
#Q2. Indexing and searching
#Index the documents with minsearch - make content a text field and filename a keyword field. Then search with this query:

#How does the agentic loop keep calling the model until it stops?

In [28]:
from minsearch import Index

# 1. Creamos el nuevo índice configurando 'content' como campo de texto y 'filename' como palabra clave
homework_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

# 2. Indexamos la lista de lecciones que descargamos en el paso anterior
homework_index.fit(documents)

# 3. Realizamos la búsqueda en el nuevo índice
search_results = homework_index.search(
    query="How does the agentic loop keep calling the model until it stops?",
)

# 4. Mostramos los resultados obtenidos para responder la pregunta
for i, doc in enumerate(search_results, 1):
    print(f"Archivo: {doc['filename']}")


Archivo: 01-agentic-rag/lessons/14-agentic-loop.md
Archivo: 01-agentic-rag/lessons/15-frameworks.md
Archivo: 01-agentic-rag/lessons/13-function-calling.md
Archivo: 01-agentic-rag/lessons/11-agents-intro.md
Archivo: 01-agentic-rag/lessons/16-other-frameworks.md
Archivo: 01-agentic-rag/lessons/12-rag-revision.md
Archivo: 01-agentic-rag/lessons/01-intro.md
Archivo: 05-monitoring/lessons/01-intro.md
Archivo: 04-evaluation/lessons/11-evaluation-intro.md
Archivo: 05-monitoring/lessons/02-assistant-setup.md


In [29]:
#Q3. RAG
#Now we will build a RAG assistant on top of this data. Let's use the rag helper script we prepared during the lessons:

INSTRUCTIONS = '''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
'''.strip()

PROMPT_TEMPLATE = '''
QUESTION: {question}

CONTEXT:
{context}
'''.strip()


class HomeworkRAG:

    def __init__(
        self,
        index,
        llm_client,
        instructions=INSTRUCTIONS,
        prompt_template=PROMPT_TEMPLATE,
        model='gemini-2.5-flash-lite'  
    ):
        self.index = index
        self.llm_client = llm_client
        self.instructions = instructions
        self.prompt_template = prompt_template
        self.model = model

    def search(self, query, num_results=5):
        # Adaptado a la tarea: Buscamos directamente en el índice minsearch de lecciones
        return self.index.search(
            query=query,
            num_results=num_results
        )

    def build_context(self, search_results):
        lines = []
        # Adaptado al nuevo esquema de la tarea (filename y content)
        for doc in search_results:
            lines.append(f"File: {doc['filename']}")
            lines.append(f"Content:\n{doc['content']}")
            lines.append("\n" + "="*40 + "\n")

        return "\n".join(lines).strip()

    def build_prompt(self, query, search_results):
        context = self.build_context(search_results)
        return self.prompt_template.format(
            question=query, context=context
        )

    def llm(self, prompt):
        # Adaptado a Gemini nativo: Envia las instrucciones y retorna la respuesta completa
        response = self.llm_client.models.generate_content(
            model=self.model,
            contents=prompt,
            config={
                'system_instruction': self.instructions
            }
        )
        return response

    def rag(self, query):
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
        
        # Obtenemos la respuesta completa de la API de Google
        response = self.llm_client.models.generate_content(
            model=self.model,
            contents=prompt,
            config={
                'system_instruction': self.instructions
            }
        )
        
        # Extraemos el texto y los metadatos de consumo
        answer = response.text
        usage = response.usage_metadata
        
        # Retornamos ambos elementos como pide la tarea
        return answer, usage


In [30]:
# 1. Instanciamos el asistente adaptado con tu cliente nativo de Gemini
assistant = HomeworkRAG(
    index=homework_index,      
    llm_client=gemini_client,  
)

# 2. Ejecutamos la consulta y desempaquetamos la tupla (answer, usage)
query_tarea = "How does the agentic loop keep calling the model until it stops?"
answer, usage = assistant.rag(query_tarea)

# 3. Imprimimos los resultados solicitados por la tarea
print("=== RESPUESTA DEL ASISTENTE ===")
print(answer)
print("-" * 50)
print(f"Tokens de entrada (Prompt Tokens): {usage.prompt_token_count}")
print(f"Tokens de salida (Completion Tokens): {usage.candidates_token_count}")


=== RESPUESTA DEL ASISTENTE ===
The agentic loop keeps calling the model in a `while` loop. This loop continues as long as the model's response includes function calls. The loop terminates when the model provides a response that contains no function calls, indicating that it has finished its task and is ready to provide a final answer.
--------------------------------------------------
Tokens de entrada (Prompt Tokens): 7981
Tokens de salida (Completion Tokens): 61


In [31]:
#Q4. Chunking
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [33]:
print(len(chunks))

295


In [36]:
#Q5. RAG with chunking
#Chunking makes each request smaller, because we send a smaller context to the LLM. Let's measure that.

#Index the chunks from Q4 (same as before: content as a text field, filename as a keyword field), point your RAG at the chunk index, and answer the same query again - reading the input tokens the same way as in Q3.

In [40]:
import json

# 2. Creamos un NUEVO índice exclusivo para los chunks (Requisito de Q5)
chunk_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
chunk_index.fit(chunks)

# 3. Re-instanciamos nuestro asistente RAG apuntando al NUEVO índice de fragmentos
assistant_chunks = HomeworkRAG(
    index=chunk_index,       
    llm_client=gemini_client,   
)

# 4. Lanzamos la misma consulta exacta para medir el impacto en la memoria

answer, usage = assistant_chunks.rag("How does the agentic loop keep calling the model until it stops?")

# 5. Imprimimos el veredicto final de los tokens consumidos
print("\n" + "="*50)
print(f"Tokens de entrada (Prompt Tokens) enviados con Chunks: {usage.prompt_token_count}")
print(f"Tokens de salida (Completion Tokens) generados: {usage.candidates_token_count}")
print("="*50)



Tokens de entrada (Prompt Tokens) enviados con Chunks: 2633
Tokens de salida (Completion Tokens) generados: 102


In [67]:
#Q6. Turning it into an agent
homework_search_counter = 0

def search_chunks(query: str) -> list:
    """
    Search the course lessons database for chunks matching the given query.
    """
    global homework_search_counter
    homework_search_counter += 1  # Cada vez que Gemini la use
    return chunk_index.search(query=query, num_results=5)

# 2. Instrucciones
instructions_q6 = """
You're a course teaching assistant. Answer the student's question using the search tool. 
Make multiple searches with different keywords before answering.
""".strip()

# 3. Llamada al Agente con la herramienta integrada de forma nativa
pregunta_tarea = "How does the agentic loop work, and how is it different from plain RAG?"

response = gemini_client.models.generate_content(
    model='gemini-2.5-flash',
    contents=pregunta_tarea,
    config={
        'tools': [search_chunks],  # ¡Google lee e indexa tu función automáticamente!
        'system_instruction': instructions_q6
    }
)

# 4. El veredicto final de la tarea
print(f"=== RESPUESTA FINAL ===\n{response.text}\n" + "-"*50)
print(f"El agente llamó a la función de búsqueda un total de: {homework_search_counter} veces.")


=== RESPUESTA FINAL ===
The agentic loop is a dynamic process where an AI agent repeatedly interacts with an LLM, executes tools, and processes the results until a final answer is reached. This "while loop" pattern allows the agent to make decisions and adapt its actions based on the current goal and information, especially when the exact sequence of steps isn't known beforehand or when dynamic information necessitates changes in the workflow. It involves managing conversation history and can entail multiple calls to the LLM and various tools.

In contrast, plain Retrieval Augmented Generation (RAG) is a more linear, three-step pipeline:
1.  **Search:** Retrieve relevant information based on a query (e.g., using keyword or vector search).
2.  **Build Prompt:** Construct a prompt that includes the original question and the retrieved information.
3.  **LLM Call:** Send the combined prompt to the LLM to generate a response.

The core difference lies in the *control flow* and *adaptability